# 1.bind函数使用技巧

知识点讲解：Runnable.bind() 动态绑定默认调用参数

1. bind() 方法的核心作用
   - 为 Runnable 组件预先绑定（固定）一部分调用参数
   - 返回一个新的 RunnableBinding 对象，原组件保持不变（不可变设计）
   - 绑定的参数会在每次 invoke 时自动作为 kwargs 传入底层组件

2. 为什么需要 bind()：设计动机
   - LCEL 管道中组件通过 | 连接，调用时只能传递单一的数据流
   - 无法在管道中途为某个组件补充额外的调用参数
   - bind() 提前把这些参数「焊死」在组件上，解决参数传递断层问题
   - 典型场景：
     * 场景 1：构建一个通用 LLM，在不同链里才绑定不同停止词
     * 场景 2：同一 LLM 派生高确定性链（temperature≈0.7）与高创造性链（temperature≈1.2）
     * 场景 3：为链中某 LLM 固定绑定工具或函数调用能力
   - 灵活性优势：无需实例化多个 LLM 对象，从同一个基础 LLM 派生即可

3. bind() 在 LLM 场景的典型用途
   - 覆盖模型参数：model、temperature、max_tokens、top_p
   - 绑定停止词：llm.bind(stop="world")
   - 绑定工具/函数调用：llm.bind(tools=[...])、llm.bind(functions=[...])
   - 绑定响应格式：llm.bind(response_format={"type": "json_object"})

4. 典型输出示例：停止词绑定的效果
   - 构建 prompt | llm.bind(stop="world") | StrOutputParser()，传入 {"query": "Hello world"}
   - 模型在生成到 "world" 处即停止，最终输出：
       Hello
   - 观察点：stop 参数在调用时自动注入 LLM 请求，无需在 invoke 时手动传，链的正常调用方式完全不变

5. bind() 与构造参数的优先级关系
   - 构造时 ChatOpenAI(model="A")，bind 时 .bind(model="B")
   - 最终请求使用 B，因为 bind 的参数在调用时覆盖构造时的默认值
   - 这使得同一个 LLM 实例可以派生出多个不同配置的变体

6. bind() 的底层实现原理
   - bind() 本质上是往 Runnable 的 kwargs 属性添加对应的字段
   - 生成一个新的 RunnableBinding 对象，包装原 Runnable 和绑定的 kwargs
   - 当 Runnable 组件执行调用时（invoke、stream、batch、ainvoke 等），会自动将
     kwargs 字段里的所有参数合并并覆盖默认调用参数
   - 从而完成动态添加默认调用参数的效果

7. bind() 与 configurable_fields() 的区别
   - bind()：构建期固定参数，构建链时确定，运行时不可再改
   - configurable_fields()：声明可配置字段，运行时通过 config 动态传值
   - 选择原则：固定不变用 bind，需按请求切换用 configurable_fields

8. 不可变性与链复用
   - bind() 不修改原对象，因此可以从同一个 llm 派生多个变体
   - 例如：fast_llm = llm.bind(model="flash")、pro_llm = llm.bind(model="pro")
   - 这些变体可分别用于不同的链，互不干扰

9. bind() 的适用边界
   - 虽然 bind() 是所有 Runnable 共有的方法，但并非所有组件都支持绑定默认调用参数
   - 部分组件底层没有默认调用参数的概念，例如 PromptTemplate 底层的 invoke 方法：
       def invoke(self, input, config=None) -> PromptValue:
           config = ensure_config(config)
           if self.metadata: config["metadata"] = {...}
           if self.tags: config["tags"] = config["tags"] + self.tags
           return self._call_with_config(self._format_prompt_with_error_handling, input, config, run_type="prompt")
   - 可见其 invoke 直接处理 input，并不消费额外的 kwargs，使用前需确认目标组件的 invoke 是否支持接收额外 kwargs

10. 数据流转过程
   - {"query": str} → prompt → Messages
   - Messages → RunnableBinding(llm, kwargs={"model": "deepseek-v4-pro"})
     → 实际请求：llm.invoke(Messages, model="deepseek-v4-pro") → AIMessage
   - AIMessage → StrOutputParser → str


In [ ]:
import dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
prompt = ChatPromptTemplate.from_messages([
    ("human", "{query}")
])
llm = ChatOpenAI(model="deepseek-flash")
chain = prompt | llm.bind(model="deepseek-v4-pro") | StrOutputParser()
content = chain.invoke({"query": "你是什么模型呢？"})
print(content)